# 01 - Exploratory Data Analysis

**Business question:** what does the statistical behaviour of crypto returns look like, and what does that imply for how we should model and risk-manage a portfolio?

| # | Question | Why it matters |
|---|----------|----------------|
| 1 | Are returns normally distributed? | Determines whether standard risk formulas (mean/std, Gaussian VaR) are safe |
| 2 | Does volatility cluster? | Volatility is forecastable even when direction is not |
| 3 | How correlated are coins, and when does that change? | Diversification only works if correlations are low *when you need them* |
| 4 | Is there a day-of-week effect? | A classic claim - easy to test properly |
| 5 | Does trading volume relate to price moves? | Volume is often used as a signal |
| 6 | Does market sentiment (Fear & Greed) lead returns? | Tests a popular narrative with lagged correlation and Granger causality |

Each section ends with an interpretation cell. **Read the numbers, then write your own conclusion** - the final cell prints computed facts to anchor your write-up.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "analytics").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from analytics import data, metrics, stats_tests, forecasting
from analytics.portfolio import xirr, fifo_realized_pnl

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 4.5)
pd.options.display.float_format = "{:,.4f}".format

In [ ]:
ohlcv = data.load_ohlcv()
prices = ohlcv.pivot(index="date", columns="symbol", values="close").sort_index()
volume = ohlcv.pivot(index="date", columns="symbol", values="volume").sort_index()
sources = data.data_sources()
print("Data source(s):", sources, "|", prices.shape[1], "coins |", prices.index.min().date(), "->", prices.index.max().date())
if "synthetic" in sources:
    print("\n*** WARNING: SYNTHETIC DEMO DATA. Re-run `python -m analytics.etl.ingest` for real data before reading anything into the results. ***")

## Data coverage and quality

In [ ]:
coverage = ohlcv.groupby("symbol").agg(first_day=("date", "min"), last_day=("date", "max"), days=("date", "count"))
coverage["expected_days"] = (coverage["last_day"] - coverage["first_day"]).dt.days + 1
coverage["missing_days"] = coverage["expected_days"] - coverage["days"]
display(coverage)

import json
report_path = ROOT / "analytics" / "reports" / "data_quality.json"
if report_path.exists():
    quality = json.loads(report_path.read_text())["coins"]
    display(pd.DataFrame({s: {**q["errors"], **q["warnings"]} for s, q in quality.items()}).T)

## 1. Are returns normally distributed?

In [ ]:
log_ret = metrics.log_returns(prices)
rows = {}
for s in log_ret.columns:
    r = log_ret[s].dropna()
    jb_stat, jb_p = stats.jarque_bera(r)
    rows[s] = {"mean_daily": r.mean(), "std_daily": r.std(), "skew": stats.skew(r),
               "excess_kurtosis": stats.kurtosis(r), "jarque_bera_p": jb_p,
               "worst_day": r.min(), "best_day": r.max(),
               "worst_day_in_std": (r.min() - r.mean()) / r.std()}
dist = pd.DataFrame(rows).T
display(dist)

focus = "BTC" if "BTC" in log_ret.columns else log_ret.columns[0]
r = log_ret[focus].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(r, bins=80, density=True, alpha=.7, label="observed")
grid = np.linspace(r.min(), r.max(), 300)
axes[0].plot(grid, stats.norm.pdf(grid, r.mean(), r.std()), "r", lw=2, label="normal with same mean/std")
axes[0].set(title=f"{focus} daily log returns", xlabel="log return"); axes[0].legend()
stats.probplot(r, dist="norm", plot=axes[1]); axes[1].set_title("Q-Q plot vs normal")
plt.tight_layout(); plt.show()

> **Interpretation:** *(write 2-3 sentences: how fat are the tails? what does a worst day of N standard deviations mean for Gaussian risk models?)*

## 2. Does volatility cluster?

In [ ]:
from statsmodels.tsa.stattools import acf
acf_raw = acf(r, nlags=30)
acf_abs = acf(r.abs(), nlags=30)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
axes[0].bar(range(1, 31), acf_raw[1:]); axes[0].set_title(f"{focus}: autocorrelation of returns")
axes[1].bar(range(1, 31), acf_abs[1:], color="C1"); axes[1].set_title(f"{focus}: autocorrelation of |returns|")
for ax in axes:
    ax.axhline(1.96 / np.sqrt(len(r)), ls="--", c="grey"); ax.axhline(-1.96 / np.sqrt(len(r)), ls="--", c="grey")
    ax.set_xlabel("lag (days)")
plt.tight_layout(); plt.show()

vol30 = log_ret.rolling(30).std() * np.sqrt(365)
vol30[[c for c in ["BTC", "ETH", "SOL"] if c in vol30.columns]].plot(title="30-day annualised volatility")
plt.ylabel("volatility"); plt.show()

> **Interpretation:** *(returns are ~unpredictable but their size is predictable - what does that mean for choosing what to forecast?)*

## 3. Correlation structure

In [ ]:
corr = metrics.correlation_matrix(log_ret.dropna())
avg_corr = metrics.average_pairwise_correlation(corr)
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
plt.title(f"Return correlation (average pairwise = {avg_corr:.2f})"); plt.show()

# Do correlations rise when the market falls?  Split days by BTC's return.
aligned = log_ret.dropna()
btc = aligned["BTC"] if "BTC" in aligned.columns else aligned.iloc[:, 0]
worst = btc <= btc.quantile(0.10)
corr_down = metrics.average_pairwise_correlation(aligned[worst].corr())
corr_other = metrics.average_pairwise_correlation(aligned[~worst].corr())
print(f"Average pairwise correlation on BTC's worst 10% of days: {corr_down:.2f}  |  all other days: {corr_other:.2f}")

if "BTC" in aligned.columns:
    roll = pd.DataFrame({s: metrics.rolling_correlation(aligned[s], aligned["BTC"], 90) for s in aligned.columns if s != "BTC"})
    roll.mean(axis=1).plot(title="Average 90-day rolling correlation with BTC"); plt.show()

> **Interpretation:** *(if correlations jump in sell-offs, how much diversification do you really get when it matters?)*

## 4. Day-of-week effect

In [ ]:
table, h_stat, dow_p = stats_tests.day_of_week_test(log_ret[focus])
display(table)
table["mean"].plot.bar(title=f"{focus}: mean daily log return by weekday"); plt.show()
print(f"Kruskal-Wallis H = {h_stat:.2f}, p = {dow_p:.3f}")

# Multiple testing: across many coins, ~5% will look 'significant' by pure chance.
pvals = {s: stats_tests.day_of_week_test(log_ret[s])[2] for s in log_ret.columns}
n_sig = sum(p < 0.05 for p in pvals.values())
print(f"{n_sig} of {len(pvals)} coins show p < 0.05 (about {0.05 * len(pvals):.1f} expected by chance alone)")

> **Interpretation:** *(is there a real weekday effect, or just noise? note the multiple-testing caveat above)*

## 5. Volume and price moves

In [ ]:
common = log_ret.dropna().index.intersection(volume.dropna(how="all").index)
vol_rows = {}
for s in log_ret.columns:
    ar = log_ret.loc[common, s].abs()
    lv = np.log(volume.loc[common, s].replace(0, np.nan))
    vol_rows[s] = {"corr(|return|, log volume) same day": ar.corr(lv, method="spearman"),
                   "corr(|return| today, log volume yesterday)": ar.corr(lv.shift(1), method="spearman")}
volume_corr = pd.DataFrame(vol_rows).T
display(volume_corr)

> **Interpretation:** *(does high volume coincide with big moves? does it *predict* them?)*

## 6. Does sentiment lead returns?

In [ ]:
try:
    fng = data.load_fear_greed()
    sent = pd.concat([fng["value"].rename("fng"), log_ret[focus].rename("ret")], axis=1).dropna()
    sent["fng_change"] = sent["fng"].diff()

    lag_corr = stats_tests.lagged_correlation(sent["fng_change"], sent["ret"], max_lag=10)
    lag_corr.plot.bar(title=f"corr(sentiment change at t-k, {focus} return at t)"); plt.xlabel("k (days sentiment leads)"); plt.show()

    g_s2r = stats_tests.granger_pvalues(sent["ret"], sent["fng_change"].fillna(0), 5)
    g_r2s = stats_tests.granger_pvalues(sent["fng_change"].fillna(0), sent["ret"], 5)
    display(pd.DataFrame({"sentiment -> returns (p)": g_s2r, "returns -> sentiment (p)": g_r2s}))

    sent["next_day_ret"] = sent["ret"].shift(-1)
    bucket = pd.cut(sent["fng"], [-1, 24, 44, 55, 75, 100], labels=["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"])
    by_bucket = sent.groupby(bucket, observed=True)["next_day_ret"].agg(["mean", "std", "count"])
    display(by_bucket)
    sentiment_ok = True
except Exception as error:
    sentiment_ok = False
    print("Sentiment analysis skipped:", error)

> **Interpretation:** *(Granger tests whether sentiment adds predictive information beyond past returns. Does it, or is sentiment just reacting to price?)*

## Key takeaways (computed from the data above)

In [ ]:
lines = []
lines.append(f"- **Fat tails:** {focus} excess kurtosis = {dist.loc[focus, 'excess_kurtosis']:.1f} (a normal distribution = 0); worst day was {dist.loc[focus, 'worst_day_in_std']:.1f} standard deviations from the mean.")
lines.append(f"- **Volatility persistence:** autocorrelation of |returns| at lag 1 is {acf_abs[1]:.2f} versus {acf_raw[1]:.2f} for raw returns (a clear gap in favour of |returns| indicates volatility clustering).")
lines.append(f"- **Correlation:** average pairwise correlation is {avg_corr:.2f} overall, but {corr_down:.2f} on {focus}'s worst 10% of days versus {corr_other:.2f} on other days.")
verdict = "significant" if dow_p < 0.05 else "no statistically significant"
lines.append(f"- **Day-of-week:** {focus} shows {verdict} weekday differences (Kruskal-Wallis p = {dow_p:.3f}); {n_sig}/{len(pvals)} coins pass p<0.05 vs ~{0.05*len(pvals):.1f} expected by chance.")
lines.append(f"- **Volume:** mean same-day rank correlation between |return| and volume is {volume_corr.iloc[:, 0].mean():.2f}.")
if sentiment_ok:
    lines.append(f"- **Sentiment:** smallest Granger p-value across 5 lags for sentiment -> returns is {g_s2r.min():.3f}; for returns -> sentiment it is {g_r2s.min():.3f}. (Taking the minimum over several lags inflates false positives - treat values near 0.05 with caution.)")
from IPython.display import Markdown
Markdown("\n".join(lines))